# Logistic Regression for Type 2 Diabetes Risk

This notebook trains a **binary logistic regression model** using only NumPy/Pandas on:

`x = [EXERANY2, _BMI5CAT, _SMOKER3, MENTHLTH, SSBFRUT3, ALCDAY4]`

Target definition:
- `y = 1`: presence of Type 2 diabetes risk (`DIABETE4 == 1`)
- `y = 0`: absence of Type 2 diabetes risk (`DIABETE4 == 3`)

The train/test split mirrors the Naive Bayes setup (`test_size=0.2`, `random_state=1`).


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Load and prepare data
feature_cols = ['EXERANY2', '_BMI5CAT', '_SMOKER3', 'MENTHLTH', 'SSBFRUT3', 'ALCDAY4']
target_col = 'DIABETE4'

df = pd.read_csv('diabetes_dataset.csv')
df = df[df[target_col].isin([1, 3])].copy()

# Keep only the requested feature vector + target
df = df[feature_cols + [target_col]].dropna().copy()

X = df[feature_cols].astype(float).to_numpy()
y_raw = df[target_col].to_numpy()

# Problem formulation: y in {0,1}
# 1 -> diabetes risk present, 3 -> diabetes risk absent
y = (y_raw == 1).astype(float)

print('Rows used:', len(df))
print('Positive class rate (y=1):', y.mean())

In [ ]:
def manual_train_test_split(X, y, test_size=0.2, random_state=1):
    """
    Reproduce sklearn-style shuffled split settings used in Naive_Bayes:
    test_size=0.2, random_state=1.
    """
    n = X.shape[0]
    n_test = int(np.ceil(test_size * n))

    rng = np.random.RandomState(random_state)
    indices = rng.permutation(n)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = manual_train_test_split(
    X, y, test_size=0.2, random_state=1
)

print('Train size:', X_train.shape[0])
print('Test size :', X_test.shape[0])

In [ ]:
# Standardize features using TRAIN statistics only
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
sigma[sigma == 0] = 1.0

X_train_std = (X_train - mu) / sigma
X_test_std = (X_test - mu) / sigma

print('Feature means (train, standardized):', X_train_std.mean(axis=0).round(4))
print('Feature stds  (train, standardized):', X_train_std.std(axis=0).round(4))

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(y_true, y_prob, eps=1e-12):
    y_prob = np.clip(y_prob, eps, 1.0 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1.0 - y_true) * np.log(1.0 - y_prob))


def fit_logistic_regression(X, y, lr=0.05, n_iter=5000, verbose_every=500):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    history = []

    for i in range(n_iter):
        z = X @ w + b
        p = sigmoid(z)

        grad_w = (X.T @ (p - y)) / n_samples
        grad_b = np.mean(p - y)

        w -= lr * grad_w
        b -= lr * grad_b

        if i % verbose_every == 0 or i == n_iter - 1:
            loss = binary_cross_entropy(y, p)
            history.append((i, loss))

    return w, b, history


def predict_proba(X, w, b):
    return sigmoid(X @ w + b)


def predict_label(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


def confusion_counts(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return {'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}

In [ ]:
# Train model
w, b, history = fit_logistic_regression(
    X_train_std,
    y_train,
    lr=0.05,
    n_iter=5000,
    verbose_every=500,
)

print('Final bias:', round(float(b), 6))
print('Loss history (iteration, loss):')
for it, loss in history:
    print(f'  {it:5d}: {loss:.6f}')

# Evaluate
yhat_train = predict_label(X_train_std, w, b, threshold=0.5)
yhat_test = predict_label(X_test_std, w, b, threshold=0.5)

train_acc = accuracy(y_train, yhat_train)
test_acc = accuracy(y_test, yhat_test)

print('\nTrain accuracy:', round(float(train_acc), 4))
print('Test accuracy :', round(float(test_acc), 4))

print('\nTrain confusion:', confusion_counts(y_train.astype(int), yhat_train))
print('Test confusion :', confusion_counts(y_test.astype(int), yhat_test))

In [ ]:
# Coefficient table (on standardized features)
coef_table = pd.DataFrame({
    'feature': feature_cols,
    'weight': w,
    'abs_weight': np.abs(w),
}).sort_values('abs_weight', ascending=False)

coef_table